# AI Health Companion: Missed-Dose Risk Predictor
Predicts the chance that a patient misses the next dose. Data is **synthetic** because real patient data is private. Run all cells top to bottom.

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

## 1. Create the synthetic dataset

In [ ]:
rng=np.random.default_rng(42); n=6000
df=pd.DataFrame({'age':rng.integers(50,91,n),'doses_per_day':rng.integers(1,9,n),
 'is_night':rng.integers(0,2,n),'is_weekend':rng.integers(0,2,n),
 'adherence_7d':np.clip(rng.normal(0.75,0.18,n),0.1,1.0),
 'lives_alone':rng.integers(0,2,n),'missed_yesterday':rng.integers(0,2,n)})
z=(-2.6+0.03*(df.age-65)+0.30*(df.doses_per_day-3)+0.8*df.is_night+0.5*df.is_weekend
   +0.7*df.lives_alone+1.0*df.missed_yesterday-3.5*(df.adherence_7d-0.75)+rng.normal(0,0.5,n))
df['missed']=(rng.random(n)<1/(1+np.exp(-z))).astype(int)
df.head()

## 2. Train and evaluate

In [ ]:
F=list(df.columns[:-1]); X,y=df[F],df['missed']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
sc=StandardScaler().fit(Xtr)
lr=LogisticRegression(max_iter=1000).fit(sc.transform(Xtr),ytr)
rf=RandomForestClassifier(200,max_depth=8,random_state=42).fit(Xtr,ytr)

In [ ]:
def met(m,Xt):
    p=m.predict(Xt);pr=m.predict_proba(Xt)[:,1]
    return {'accuracy':round(accuracy_score(yte,p),3),'precision':round(precision_score(yte,p),3),'recall':round(recall_score(yte,p),3),'auc':round(roc_auc_score(yte,pr),3)}
print('Logistic Regression:',met(lr,sc.transform(Xte)))
print('Random Forest:',met(rf,Xte))

## 3. Feature importance (Logistic Regression)

In [ ]:
pd.Series(lr.coef_[0],index=F).sort_values().plot(kind='barh',title='Effect on miss risk (positive = raises risk)')